In [6]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
# import threading
mt5.initialize()


True

In [7]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [50]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 3, 1, tzinfo=timezone)
    utc_to = datetime(x.year, x.month+1, x.day+1, tzinfo=timezone)
    # utc_to = datetime(x.year, x.month+1, 1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_H1, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    
#     rates_frame['mean'] = (rates_frame['high'] + rates_frame['low'])/2

    # EMA = rates_frame['close'].ewm(span=100, adjust=False).mean()
    # DEMA = 2*EMA - EMA.ewm(span=100, adjust=False).mean()
    # rates_frame['dma'] = DEMA
#     rates_frame['P34'] = rates_frame['mean'].rolling(window=34).mean()
#     rates_frame['P5'] = rates_frame['mean'].rolling(window=5).mean()
#     rates_frame = rates_frame.fillna(0)
#     rates_frame['AO'] = rates_frame['P5'] - rates_frame['P34']
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    rates_frame['rsi'] = pta.rsi(a['close'], length = 14)
    
    return rates_frame

In [51]:
a = get_values("EURGBP")


In [61]:
a.iloc[0].open-a.iloc[0].close

-0.0006000000000000449

In [56]:
p = []
check = 1
t = []
c = []
o = []
for i in range(1,len(a)-1):
    if str(a.iloc[i].rsi) != "nan":
        if a.iloc[i].rsi < 30 and check == 1:
            buy_price = a.iloc[i].close
            sell_price = a.iloc[i+1].close
            p.append(price_action("EURUSD", 1.0, buy_price, sell_price, mt5.ORDER_TYPE_SELL))
            t.append(a.iloc[i].name)
            c.append(a.iloc[i].rsi)
            o.append(a.iloc[i].open-a.iloc[i].close)
            
            check = 0
        if check == 0:
            if a.iloc[i].rsi > 30:
                check = 1

In [57]:
sum(p)

705.0

In [63]:
for i in range(0, len(p)):
    print(p[i],"--", c[i], "---", t[i],"***", o[i])

88.0 -- 29.80854902583598 --- 2021-03-04 15:00:00 *** 0.0005299999999999194
38.0 -- 28.923889100188184 --- 2021-03-08 11:00:00 *** 0.0010700000000000154
-164.0 -- 29.83746529762781 --- 2021-03-17 09:00:00 *** 0.0014100000000000223
-123.0 -- 27.675443151614363 --- 2021-03-18 12:00:00 *** 0.0016199999999999548
160.0 -- 24.65764225342692 --- 2021-03-25 17:00:00 *** 0.0018899999999999473
1.0 -- 26.189729783099814 --- 2021-03-26 11:00:00 *** 0.0010000000000000009
-96.0 -- 27.633403611434304 --- 2021-03-26 15:00:00 *** 0.0010700000000000154
76.0 -- 25.76992649823651 --- 2021-03-29 11:00:00 *** 0.0023100000000000342
119.0 -- 29.030171038967563 --- 2021-04-05 10:00:00 *** 0.0007900000000000684
30.0 -- 29.7933170600541 --- 2021-04-19 04:00:00 *** 0.0013100000000000334
169.0 -- 28.094686619687828 --- 2021-04-19 16:00:00 *** 0.0010900000000000354
4.0 -- 29.869984315736456 --- 2021-05-03 17:00:00 *** 0.00025999999999992696
28.0 -- 28.422472484252825 --- 2021-05-04 11:00:00 *** 0.000989999999999935

In [8]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 4, 1, tzinfo=timezone)
    utc_to = datetime(x.year, x.month+1, x.day+1, tzinfo=timezone)
    # utc_to = datetime(x.year, x.month+1, 1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_H1, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    
    rates_frame['mean'] = (rates_frame['high'] + rates_frame['low'])/2

    # EMA = rates_frame['close'].ewm(span=100, adjust=False).mean()
    # DEMA = 2*EMA - EMA.ewm(span=100, adjust=False).mean()
    # rates_frame['dma'] = DEMA
    rates_frame['P34'] = rates_frame['mean'].rolling(window=34).mean()
    rates_frame['P5'] = rates_frame['mean'].rolling(window=5).mean()
    rates_frame['sma'] = rates_frame['close'].rolling(window=100).mean()
    
#     rates_frame = rates_frame.fillna(0)
    rates_frame['AO'] = rates_frame['P5'] - rates_frame['P34']
    rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    rates_frame = rates_frame.drop(['high', 'low', 'mean', 'P34', 'P5'], axis=1)
    return rates_frame

In [9]:
symbol = "EURGBP"
a = get_values(symbol)

In [10]:
l = ['NR']
for i in range(1,len(a)):
    df = a.iloc[i].AO
    dfo = a.iloc[i-1].AO
    if str(df) == 'nan':
        l.append('NR')
    else:
        if df <= 0.0:
            if df > dfo:
                l.append('NG')
            elif df < dfo:
                l.append("NR")
            else:
                print(type(df))
                l.append('R')
        elif df > 0.0:
            if df > dfo:
                l.append('PG')
            elif df < dfo:
                l.append("PR")
            else:
                print(type(df))
                l.append('NR')
        else:
            l.append('NR')
a['signal'] = l

<class 'numpy.float64'>


In [11]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
m = 0
n = 0
p= []
checks = 0
counterr = 0
counterp = 0
counterrp = 0
profits = []
l = 0

for i in range(7, len(a)):
    df = a.iloc[i].signal
    dfo = a.iloc[i-1].signal
    if str(a.iloc[i].sma) != 'nan':        
        if a.iloc[i].open < a.iloc[i].sma and a.iloc[i].close < a.iloc[i].sma \
        and a.iloc[i-1].open > a.iloc[i].sma and a.iloc[i-1].close < a.iloc[i].sma and check == 0:
            counter = 0
            counterr = 0
            m = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
            print("SELL")
            print("*"*20)
            check = 1    

        elif a.iloc[i].open < a.iloc[i].sma and a.iloc[i].close < a.iloc[i].sma \
        and a.iloc[i-1].open < a.iloc[i].sma and a.iloc[i-1].close > a.iloc[i].sma and check == 0:
            counter = 0
            counterr = 0
            m = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*100)
            print(a.iloc[i].name)
            print("SELL")
            print("*"*100)
            check = 1    
            
            
        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            print(pp,"---",a.iloc[i].close)
#             profit.append(pp)
#             check = 0
            
            if pp > 0.0:
                counterr = counterr + 1
                if counterr > 2:
                    profit.append(pp)
                    check = 0
                    print(f"checks--> {checks}")
            if pp > 0.0 and m == 1:
                profit.append(pp)
                m = 0
            elif a.iloc[i].close > a.iloc[i].sma:
                if m == 1:
                    print("+"*100)
                    profit.append(2*pp)
                    check = 0
                    print(f"checks--> {checks}")
                else:
                    profit.append(pp)
                    check = 0  
                    print(f"checks--> {checks}")

#             elif pp < 0.0:
#                 counter = counter + 1
#                 if counter > 3:
#                     if m == 1:
#                         print("+"*100)
#                         profit.append(2*pp)
#                         checks = 0
#                     else:
#                         profit.append(pp)
#                         checks = 0         

        if a.iloc[i].open > a.iloc[i].sma and a.iloc[i].close > a.iloc[i].sma \
        and a.iloc[i-1].open < a.iloc[i].sma and a.iloc[i-1].close > a.iloc[i].sma and checks == 0:
            counterp = 0
            counterrp = 0
            n = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
            print("BUY")
            print("*"*20)
            checks = 1   
            
        elif a.iloc[i].open > a.iloc[i].sma and a.iloc[i].close > a.iloc[i].sma \
         and a.iloc[i-1].open > a.iloc[i].sma and a.iloc[i-1].close < a.iloc[i].sma \
         and checks == 0:
            counterp = 0
            counterrp = 0
            n = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*100)
            print(a.iloc[i].name)
            print("BUY")
            print("*"*100)
            checks = 1   

        elif checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---",a.iloc[i].close)
#             profit.append(pp)
#             check = 0
            
            if pp > 0.0:
                counterrp = counterrp + 1
                if counterrp > 2:
                    profit.append(pp)
                    checks = 0
                    print(f"check--> {check}")
            if pp > 0.0 and n == 1:
                profit.append(pp)
                n = 0
            elif a.iloc[i].close < a.iloc[i].sma:
                if n == 1:
                    print("+"*100)
                    profit.append(2*pp)
                    checks = 0
                    print(f"check--> {check}")
                else:
                    profit.append(pp)
                    checks = 0   
                    print(f"check--> {check}")

            
#             elif pp < 0.0:
#                 counterp = counterp + 1
#                 if counter > 3:
#                     if n == 1:
#                         print("+"*100)
#                         profit.append(2*pp)
#                         checks = 0
#                     else:
#                         profit.append(pp)
#                         checks = 0         

####################
2021-04-13 09:00:00
SELL
********************
-3.21 --- 0.86565
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
checks--> 0
####################
2021-04-13 12:00:00
SELL
********************
-4.74 --- 0.86581
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
checks--> 0
####################
2021-04-13 14:00:00
BUY
********************
4.74 --- 0.86847
3.24 --- 0.86793
5.22 --- 0.86864
check--> 0
####################
2021-04-15 17:00:00
BUY
********************
0.89 --- 0.86813
1.34 --- 0.86829
0.7 --- 0.86806
check--> 0
####################
2021-04-16 01:00:00
BUY
********************
0.11 --- 0.86834
0.22 --- 0.86838
2.15 --- 0.86907
check--> 0
####################
2021-04-16 18:00:00
SELL
********************
-0.22 --- 0.86655
1.56 --- 0.86591
1.8399999999999999 --- 0.86581
2.01 --- 0.86575
checks--> 0
####################
2021-04-22 11:00:00
BUY
**************

####################
2021-06-22 13:00:00
SELL
********************
3.71 --- 0.85565
1.23 --- 0.85654
0.64 --- 0.85675
checks--> 0
####################
2021-06-24 15:00:00
BUY
********************
-0.84 --- 0.85802
0.06 --- 0.85834
-2.07 --- 0.85758
-3.54 --- 0.85705
-4.24 --- 0.8568
-4.6899999999999995 --- 0.85664
-5.8 --- 0.85624
check--> 0
####################
2021-06-30 13:00:00
SELL
********************
0.11 --- 0.85767
-1.79 --- 0.85835
-0.47 --- 0.85788
-1.8399999999999999 --- 0.85837
-1.4 --- 0.85821
-2.29 --- 0.85853
-1.2 --- 0.85814
-1.65 --- 0.8583
1.23 --- 0.85727
1.53 --- 0.85716
checks--> 0
####################
2021-07-01 12:00:00
BUY
********************
0.28 --- 0.86045
-0.92 --- 0.86002
-0.87 --- 0.86004
-1.0 --- 0.85999
2.23 --- 0.86115
0.22 --- 0.86043
check--> 0
####################
2021-07-02 19:00:00
SELL
********************
-0.5 --- 0.85812
0.53 --- 0.85775
0.98 --- 0.85759
0.95 --- 0.8576
checks--> 0
####################
2021-07-07 13:00:00
SELL
****************

In [12]:
sum(profit)

-8.87

In [13]:
for i in range(len(profit)):
    if profit[i]!=0:
        print(profit[i])

-6.42
-9.48
4.74
5.22
0.89
0.7
0.11
2.15
1.56
2.01
2.29
4.47
0.03
2.09
0.84
-3.15
1.09
-0.45
-1.5
-2.18
1.2
-3.66
-6.48
-10.88
1.81
0.98
1.9300000000000002
4.38
0.81
0.06
4.52
3.88
0.28
0.98
0.22
1.03
0.36
-1.53
0.92
2.46
0.31
0.78
0.98
4.63
0.78
0.47
1.7
0.87
-19.6
0.25
0.95
0.03
1.79
0.14
-5.9399999999999995
0.7
1.9300000000000002
1.67
4.58
1.12
1.73
0.2
-1.17
0.36
0.28
1.06
-1.2
0.39
9.71
1.23
1.62
0.45
0.75
-3.3
0.11
3.1
0.11
0.14
0.59
0.53
0.39
3.35
3.71
0.64
0.06
-5.8
0.11
1.53
0.28
0.22
0.53
0.95
0.67
0.56
3.24
6.73
2.82
3.43
0.56
-3.49
0.14
-11.02
-19.54
-7.42
-2.84
-12.62
0.42
5.36
-5.14
0.36
-0.17


In [153]:
len(profit)-14

12

In [ ]:
        if df == "PG" and "G" in dfo and a.iloc[i].rsi > 66.0 and a.iloc[i-1].rsi < 66.0 and check ==0:
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)
            check = 1    

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action("EURGBP", 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp)
#             if pp > 0.30:
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
            if 'R' in df:
                print("R")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
            elif pp < -2.0:
                print("pp")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
#             elif a.iloc[i].rsi > 80.0:
#                 print("rsi")
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#                 print(a.iloc[i].name)

            else:
                index.append(0)
                B.append('nan')
                profit.append(0)
                indexB.append(0)

In [ ]:
#             profit.append(pp)
#             print(a.iloc[i].name)
            
#             check = 0
#             if pp > 0.30:
# #                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#             if 'R' in df:
#                 print("R")
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#                 print(a.iloc[i].name)
#             if pp < -2.0:
#                 print("pp")
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#                 print(a.iloc[i].name)

In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []

for i in range(7, len(a)):
    df = a.iloc[i].signal
    dfo = a.iloc[i-1].signal
    if str(a.iloc[i].AO) != 'nan':
        aa = a.iloc[i].rsi
        b = a.iloc[i-1].rsi
        c = aa-b
        
        height = abs(a.iloc[i].open - a.iloc[i].close)
#         if "G" in df and a.iloc[i].rsi > 52.0 and a.iloc[i-1].rsi < a.iloc[i].rsi and a.iloc[i-1].rsi < 50.0 \
#         and c > 2.0  and height > 0.00070  and check ==0:
#         if a.iloc[i].rsi > 47.0 and a.iloc[i].rsi < 50.0 and a.iloc[i-1].rsi < a.iloc[i].rsi and height > 0.00030:
        if df == "NG" and a.iloc[i].rsi > 51.0 and a.iloc[i-1].rsi < 50.0:
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
#             print(a.iloc[i].rsi)
            print("*"*20)
            check = 1    

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action("EURGBP", 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---",a.iloc[i].rsi)
            if a.iloc[i].rsi < 50.0:
                print("rsi")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
            elif pp> 2.0:
                print("pp")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
                
#         if df == "PR" and a.iloc[i].rsi < 49.0 and a.iloc[i-1].rsi >= 50.0:
#             B.append("buy")
#             buy_price = a.iloc[i].close
#             indexB.append(a.iloc[i].name)
#             print("#"*20)
#             print(a.iloc[i].name)
#             print(a.iloc[i].rsi)
#             print("*"*20)
#             checks = 1    

#         elif checks == 1:
#             sell_price = a.iloc[i].close
#             pp = price_action("EURGBP", 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
#             print(pp,"---",a.iloc[i].rsi)
#             if a.iloc[i].rsi >= 50.0:
#                 print("rsi")
#                 checks = 0
#                 index.append(a.iloc[i].name)
#                 profits.append(pp)
#                 print(a.iloc[i].name)

In [89]:
q = 1
w = 2
e = 3
r = 4
if q==1 and w==2 or e==4 and r==4:
    print("df")

df
